# Tarea 1: Detección de coches, personas, matrículas.

In [2]:
%pip install pandas

Could not fetch URL https://pypi.org/simple/pandas/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/pandas/ (Caused by SSLError("Can't connect to HTTPS URL because the SSL module is not available.")) - skipping
Could not fetch URL https://pypi.org/simple/pip/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/pip/ (Caused by SSLError("Can't connect to HTTPS URL because the SSL module is not available.")) - skipping
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pandas (from versions: none)
ERROR: No matching distribution found for pandas


In [1]:
import cv2, os
import pandas as pd
from ultralytics import YOLO
from collections import defaultdict

# CONFIGURACION
VIDEO = "C0142.MP4"
MODEL = "models/yolo11s_vehiculos.pt"
PLATE_MODEL = "models/yolo11s_matriculas.pt"
OUTPUT = "results/videos/deteccion_tarea1.mp4"
CSV = "data/detecciones_tarea1.csv"

CONF = 0.35
PLATE_CONF = 0.25
IMGSZ = 960

CLASSES = [0, 2, 3, 5, 7]
NAMES = {0: "persona", 2: "coche", 3: "moto", 5: "autobus", 7: "camion"}
VEHICLES = [2, 3, 5, 7]

BOX_THICKNESS = 1
FONT_SCALE = 0.35
FONT_THICKNESS = 1

# CARGAR MODELOS
print("Cargando modelos...")
model = YOLO(MODEL)
plate_model = YOLO(PLATE_MODEL) if os.path.exists(PLATE_MODEL) else None

# ABRIR VIDEO
cap = cv2.VideoCapture(VIDEO)
if not cap.isOpened(): raise Exception("No se pudo abrir el video.")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(OUTPUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
if not out.isOpened(): raise Exception("No se pudo crear el video.")

# VARIABLES
data = []
counts = defaultdict(set)
frame_number = 0

# PROCESAMIENTO
while True:
    ret, frame = cap.read()
    if not ret: break

    frame_number += 1
    print(f"Procesando {frame_number}/{total}", end="\r")

    result = model.track(frame, persist=True, tracker="bytetrack.yaml", classes=CLASSES, conf=CONF, imgsz=IMGSZ, verbose=False)[0]
    annotated = frame.copy()

    if result.boxes is not None:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            confidence, cls = float(box.conf[0]), int(box.cls[0])
            name = NAMES[cls]
            track_id = int(box.id[0]) if box.id is not None else -1

            if track_id != -1: counts[name].add(track_id)

            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), BOX_THICKNESS)
            cv2.putText(annotated, f"{name} {track_id} {confidence:.2f}", (x1, max(y1 - 4, 12)), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (0, 255, 0), FONT_THICKNESS)

            plate_confidence = 0
            mx1 = my1 = mx2 = my2 = None

            if plate_model is not None and cls in VEHICLES:
                crop = frame[max(0, y1):min(height, y2), max(0, x1):min(width, x2)]

                if crop.size > 0:
                    plates = plate_model.predict(crop, conf=PLATE_CONF, imgsz=640, verbose=False)[0]

                    if plates.boxes is not None and len(plates.boxes):
                        best = max(plates.boxes, key=lambda b: float(b.conf[0]))
                        px1, py1, px2, py2 = map(int, best.xyxy[0].cpu().numpy())
                        mx1, my1, mx2, my2 = x1 + px1, y1 + py1, x1 + px2, y1 + py2
                        plate_confidence = float(best.conf[0])

                        cv2.rectangle(annotated, (mx1, my1), (mx2, my2), (255, 0, 0), 1)
                        cv2.putText(annotated, "PLACA", (mx1, max(my1 - 3, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 0, 0), 1)

            data.append({"fotograma": frame_number, "tipo_objeto": name, "confianza": confidence, "identificador_tracking": track_id, "x1": x1, "y1": y1, "x2": x2, "y2": y2, "confianza_matricula": plate_confidence, "mx1": mx1, "my1": my1, "mx2": mx2, "my2": my2})

    y = 20
    for name, ids in counts.items():
        cv2.putText(annotated, f"{name}: {len(ids)}", (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)
        y += 18

    out.write(annotated)

# FINALIZAR
cap.release()
out.release()
cv2.destroyAllWindows()

# EXPORTAR CSV
df = pd.DataFrame(data)
df.to_csv(CSV, index=False, encoding="utf-8-sig")

# RESULTADOS
print("\n\nPROCESAMIENTO FINALIZADO")
for name, ids in counts.items(): print(f"{name}: {len(ids)}")
print(f"\nTotal de detecciones: {len(df)}")
print(f"Video: {OUTPUT}")
print(f"CSV: {CSV}")

ModuleNotFoundError: No module named 'pandas'

# Dataset

Carpeta de Google Drive con el dataset de la práctica (`prac4VC_dataset`): https://drive.google.com/drive/folders/1RFd6sKH_QMQcpE_w5rWmy8rKiYUY5_SI

In [2]:
%pip install gdown

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)

   -------- ------------------------------- 1/5 [soupsieve]
   ------------------------ --------------- 3/5 [beautifulsoup4]
   ---------------------------------------- 5/5 [gdown]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
# Abrelo en el navegador para que veas la estructura del álbum
# Sólo interesa /images y /annotations para procesar y entrenar
# Salud y suerte :) (bro is cooking)

URL_DATASET = "https://drive.google.com/drive/folders/1RFd6sKH_QMQcpE_w5rWmy8rKiYUY5_SI"

TAREA 2

Preparación del dataset: procesa las imágenes y sus anotaciones JSON para convertirlas al formato de etiquetas TXT compatible con YOLO.

In [ ]:
import os
import json
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive
from tqdm import tqdm  # <--- Librería para la barra de progreso

# 1. Asegurar Drive montado
drive.mount('/content/drive', force_remount=False)

RUTA_BASE_DRIVE = Path('/content/drive/MyDrive/prac4VC_dataset')
DATASET_LOCAL = Path('/content/dataset_yolo')
DIR_IMAGES = DATASET_LOCAL / 'images'
DIR_LABELS = DATASET_LOCAL / 'labels'

if DATASET_LOCAL.exists():
    shutil.rmtree(DATASET_LOCAL)

DIR_IMAGES.mkdir(parents=True, exist_ok=True)
DIR_LABELS.mkdir(parents=True, exist_ok=True)

print(f"🔍 Buscando imágenes en: {RUTA_BASE_DRIVE}...")
imagenes_encontradas = list(RUTA_BASE_DRIVE.rglob('images/*.jpg')) + list(RUTA_BASE_DRIVE.rglob('images/*.jpeg')) + list(RUTA_BASE_DRIVE.rglob('images/*.JPG'))
print(f"📸 Encontradas {len(imagenes_encontradas)} imágenes. Procesando...")

def procesar_imagen(img_path):
    try:
        carpeta_padre = img_path.parent.parent
        json_path = carpeta_padre / 'annotations' / f"{img_path.stem}.json"

        if not json_path.exists():
            return False

        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if isinstance(data, dict):
            data = [data]

        txt_lines = []
        for item in data:
            plate_text = str(item.get("plate", "None"))
            if plate_text.upper() == "NONE" or not plate_text:
                continue

            rect = item.get("rectangle")
            if rect and len(rect) == 4:
                xc, yc, w, h = rect
                txt_lines.append(f"0 {xc} {yc} {w} {h}")

        if txt_lines:
            nombre_unico = f"{carpeta_padre.name}_{img_path.stem}"
            shutil.copy(img_path, DIR_IMAGES / f"{nombre_unico}{img_path.suffix}")
            (DIR_LABELS / f"{nombre_unico}.txt").write_text("\n".join(txt_lines))
            return True
    except Exception:
        pass
    return False

# Ejecutar con hilos y barra de progreso interactiva (tqdm)
contador_validas = 0
with ThreadPoolExecutor(max_workers=8) as executor:
    # Lanzamos las tareas
    futures = {executor.submit(procesar_imagen, img): img for img in imagenes_encontradas}

    # Creamos la barra de progreso que se actualiza conforme van terminando
    with tqdm(total=len(imagenes_encontradas), desc="🚀 Convirtiendo dataset") as pbar:
        for future in as_completed(futures):
            if future.result():
                contador_validas += 1
            pbar.update(1)

print(f"\n✅ ¡Proceso completado! Se han preparado {contador_validas} imágenes válidas en '{DATASET_LOCAL}'.")

Entrenamiento YOLO11: utiliza el dataset preparado para entrenar un modelo capaz de detectar matrículas y generar el archivo best.pt.

In [ ]:
import yaml
import shutil
from ultralytics import YOLO

# 1. Crear el archivo data.yaml apuntando a la carpeta limpia de Colab
data_config = {
    'path': '/content/dataset_yolo',
    'train': 'images',
    'val': 'images',
    'names': {0: 'matricula'}
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_config, f)

# 2. Cargar el modelo YOLOv11 pequeño preentrenado y empezar el entrenamiento
# epochs=30 es perfecto para esta práctica; la GPU de Colab lo hará rápido
model = YOLO("models/yolo11s_vehiculos.pt")
model.train(
    data='data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='modelo_matriculas',
    device=0
)

# 3. Guardar automáticamente el archivo best.pt en tu Google Drive principal
shutil.copy('runs/detect/modelo_matriculas/weights/best.pt', '/content/drive/MyDrive/best.pt')
print("💾 ¡Entrenamiento finalizado y copia de seguridad guardada en tu Google Drive como 'best.pt'!")

## Extras: anonimización y flujo direccional

Estos extras reutilizan `data/detecciones_tarea1.csv` y `results/videos/deteccion_tarea1.mp4`, por lo que **no vuelven a ejecutar la inferencia**. Se incluyen aquí, dentro del cuaderno, para que formen parte de la entrega.


In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE / "P4" / "data").exists():
    BASE = BASE / "P4"

CSV = BASE / "data" / "detecciones_tarea1.csv"
VIDEO = BASE / "results" / "videos" / "deteccion_tarea1.mp4"
VIDEO_OUT = BASE / "results" / "videos" / "anonimizado.mp4"
FLUJO_CSV = BASE / "results" / "flujo" / "flujo_tracks.csv"
FLUJO_RESUMEN = BASE / "results" / "flujo" / "flujo_resumen.csv"
FLUJO_PNG = BASE / "results" / "flujo" / "flujo_por_clase.png"

for carpeta in (VIDEO_OUT.parent, FLUJO_CSV.parent):
    carpeta.mkdir(parents=True, exist_ok=True)

# Las coordenadas del CSV están en el vídeo original 1920x1080.
ORIGEN_W, ORIGEN_H = 1920, 1080

# Un track se considera válido para el flujo si tiene suficiente recorrido.
MIN_DETECCIONES = 5
MIN_DESPLAZAMIENTO_PX = 20.0

CLASS_ORDER = ["coche", "persona", "camion", "autobus", "moto"]
DIR_ORDER = ["izquierda", "derecha", "arriba", "abajo"]

def pixelar(frame, x1, y1, x2, y2, bloques=10):
    """Sustituye la región por una versión pixelada."""
    x1 = max(0, int(x1))
    y1 = max(0, int(y1))
    x2 = min(frame.shape[1], int(x2))
    y2 = min(frame.shape[0], int(y2))

    if x2 - x1 < 3 or y2 - y1 < 3:
        return

    roi = frame[y1:y2, x1:x2]
    h, w = roi.shape[:2]
    small_w = max(1, w // bloques)
    small_h = max(1, h // bloques)
    pequena = cv2.resize(roi, (small_w, small_h), interpolation=cv2.INTER_LINEAR)
    pixelada = cv2.resize(pequena, (w, h), interpolation=cv2.INTER_NEAREST)
    frame[y1:y2, x1:x2] = pixelada


def anonimizar(df):
    cap = cv2.VideoCapture(str(VIDEO))
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir {VIDEO}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(VIDEO_OUT), fourcc, fps, (width, height))
    if not out.isOpened():
        raise RuntimeError(f"No se pudo crear {VIDEO_OUT}")

    sx = width / ORIGEN_W
    sy = height / ORIGEN_H

    por_fotograma = {}
    for row in df.itertuples(index=False):
        por_fotograma.setdefault(int(row.fotograma), []).append(
            (int(row.x1), int(row.y1), int(row.x2), int(row.y2))
        )

    frame_number = 0
    cajas_difuminadas = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_number += 1
        for x1, y1, x2, y2 in por_fotograma.get(frame_number, []):
            # Pequeño margen para incluir el borde de la caja.
            margen = 4
            pixelar(
                frame,
                x1 * sx - margen,
                y1 * sy - margen,
                x2 * sx + margen,
                y2 * sy + margen,
            )
            cajas_difuminadas += 1

        cv2.rectangle(frame, (10, 10), (255, 45), (0, 0, 0), -1)
        cv2.putText(
            frame,
            "ANONIMIZADO",
            (18, 36),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

        out.write(frame)
        if frame_number % 250 == 0:
            print(f"Anonimizando {frame_number}/{total}", end="\r")

    cap.release()
    out.release()
    print(f"\nVídeo anonimizado: {VIDEO_OUT}")
    print(f"Fotogramas: {frame_number} | Cajas pixeladas: {cajas_difuminadas}")


def calcular_flujo(df):
    filas = []
    for (track_id, clase), grupo in df.groupby(
        ["identificador_tracking", "tipo_objeto"], sort=False
    ):
        grupo = grupo.sort_values("fotograma")
        if len(grupo) < 2:
            continue

        primero = grupo.iloc[0]
        ultimo = grupo.iloc[-1]

        x_ini = (float(primero.x1) + float(primero.x2)) / 2.0
        y_ini = (float(primero.y1) + float(primero.y2)) / 2.0
        x_fin = (float(ultimo.x1) + float(ultimo.x2)) / 2.0
        y_fin = (float(ultimo.y1) + float(ultimo.y2)) / 2.0

        dx = x_fin - x_ini
        dy = y_fin - y_ini
        distancia = float(np.hypot(dx, dy))

        if distancia < MIN_DESPLAZAMIENTO_PX:
            direccion = "sin_direccion"
        elif abs(dx) >= abs(dy):
            direccion = "derecha" if dx > 0 else "izquierda"
        else:
            direccion = "abajo" if dy > 0 else "arriba"

        valido = len(grupo) >= MIN_DETECCIONES and distancia >= MIN_DESPLAZAMIENTO_PX

        if len(grupo) < MIN_DETECCIONES:
            motivo = "pocas_detecciones"
        elif distancia < MIN_DESPLAZAMIENTO_PX:
            motivo = "desplazamiento_pequeno"
        else:
            motivo = ""

        filas.append(
            {
                "identificador_tracking": int(track_id),
                "tipo_objeto": clase,
                "fotograma_inicio": int(primero.fotograma),
                "fotograma_fin": int(ultimo.fotograma),
                "num_detecciones": int(len(grupo)),
                "x_inicio": round(x_ini, 1),
                "y_inicio": round(y_ini, 1),
                "x_fin": round(x_fin, 1),
                "y_fin": round(y_fin, 1),
                "dx": round(dx, 1),
                "dy": round(dy, 1),
                "distancia_px": round(distancia, 1),
                "direccion": direccion,
                "valido": int(valido),
                "motivo_descarte": motivo,
            }
        )

    flujo = pd.DataFrame(filas)
    flujo = flujo.sort_values(["tipo_objeto", "identificador_tracking"]).reset_index(
        drop=True
    )
    flujo.to_csv(FLUJO_CSV, index=False, encoding="utf-8")

    validos = flujo[flujo["valido"] == 1]
    resumen = (
        validos.groupby(["tipo_objeto", "direccion"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=DIR_ORDER, fill_value=0)
    )

    totales = flujo.groupby("tipo_objeto").size().rename("total_tracks")
    validos_por_clase = validos.groupby("tipo_objeto").size().rename("tracks_validos")
    resumen = resumen.join(totales).join(validos_por_clase, how="outer").fillna(0)
    resumen["tracks_descartados"] = resumen["total_tracks"] - resumen["tracks_validos"]
    resumen = resumen.astype(int)

    orden = [c for c in CLASS_ORDER if c in resumen.index]
    orden += [c for c in resumen.index if c not in orden]
    resumen = resumen.loc[orden]
    resumen.to_csv(FLUJO_RESUMEN, encoding="utf-8")

    # Gráfica de barras agrupadas por clase y dirección.
    plot = resumen[DIR_ORDER]
    ax = plot.plot(
        kind="bar",
        figsize=(10, 5),
        color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"],
        width=0.8,
    )
    ax.set_title("Flujo direccional por clase (tracks válidos)")
    ax.set_xlabel("Clase")
    ax.set_ylabel("Número de tracks")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Dirección", ncol=4, loc="upper right")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FLUJO_PNG, dpi=160, bbox_inches="tight")
    plt.close()

    print(f"Flujo por track: {FLUJO_CSV}")
    print(f"Resumen: {FLUJO_RESUMEN}")
    print(f"Gráfica: {FLUJO_PNG}")
    print(f"Tracks totales: {len(flujo)} | válidos: {len(validos)}")

df = pd.read_csv(CSV)
print(f"Detecciones cargadas: {len(df)} filas")


### Anonimización

Se recorre el vídeo y se pixelan las regiones de personas y vehículos. La salida es `results/videos/anonimizado.mp4`.


In [ ]:
anonimizar(df)


### Flujo direccional por clase

Se analiza el desplazamiento de cada track y se descartan los que tienen pocas detecciones o poco recorrido. Las salidas son `results/flujo/flujo_tracks.csv`, `results/flujo/flujo_resumen.csv` y `results/flujo/flujo_por_clase.png`.


In [ ]:
calcular_flujo(df)

resumen = pd.read_csv(FLUJO_RESUMEN, index_col=0)
flujo = pd.read_csv(FLUJO_CSV)

from IPython.display import Image, display
print("Resumen por clase y dirección:")
display(resumen)
print("Primeros tracks:")
display(flujo.head(10))
display(Image(filename=str(FLUJO_PNG)))


Comparativa 2 modelos PaddleOCR y SmolVLM

In [ ]:

# ==============================================================
# PRACTICA 4 - COMPARATIVA OCR CLASICO VS VLM
#
# DETECTOR: YOLO (best.pt ya entrenado)
# OCR CLASICO: PaddleOCR
# VLM: SmolVLM-Instruct
#
# ENTRADA: IMAGENES
# SALIDA: CSV + GRAFICAS + METRICAS
# ==============================================================


# ==============================================================
# 1. INSTALACION
# ==============================================================

!pip install -q paddlepaddle paddleocr ultralytics pandas pillow transformers accelerate tqdm matplotlib


# ==============================================================
# 2. IMPORTACIONES Y CONFIGURACION
# ==============================================================

from google.colab import drive, files

drive.mount('/content/drive')

import os
import glob
import re
import time
import gc
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from paddleocr import PaddleOCR
from transformers import AutoProcessor, AutoModelForImageTextToText


# --------------------------------------------------------------
# RUTAS
# --------------------------------------------------------------

RUTA_BEST = '/content/drive/MyDrive/best.pt'

RUTA_DATASET = '/content/drive/MyDrive/prac4VC_dataset'

RUTA_CSV = '/content/comparativa_ocr_vlm.csv'

RUTA_METRICAS = '/content/metricas_comparativa.csv'

RUTA_GRAFICAS = '/content/graficas_comparativa'


# Si tienes las matriculas reales en un CSV, indica su ruta.
# El archivo debe tener dos columnas: imagen y matricula_real.
# Si no tienes ese archivo, deja el valor en None.

RUTA_GROUND_TRUTH = None

# Ejemplo:
# RUTA_GROUND_TRUTH = '/content/drive/MyDrive/matriculas_reales.csv'


# --------------------------------------------------------------
# PARAMETROS
# --------------------------------------------------------------

UMBRAL_YOLO = 0.50
UMBRAL_OCR = 0.30

ESCALA_OCR = 3

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('=' * 60)
print('COMPARATIVA DE RECONOCIMIENTO DE MATRICULAS')
print('=' * 60)

print('Dispositivo:', DEVICE)

if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


# Comprobar rutas

if not os.path.exists(RUTA_BEST):
    raise FileNotFoundError(
        'No se encuentra best.pt. Comprueba su ruta.'
    )

if not os.path.exists(RUTA_DATASET):
    raise FileNotFoundError(
        'No se encuentra la carpeta del dataset.'
    )


# ==============================================================
# 3. CARGAR YOLO (NO SE ENTRENA DE NUEVO)
# ==============================================================

print('\nCargando tu modelo YOLO entrenado...')

model_yolo = YOLO(RUTA_BEST)

print('YOLO cargado correctamente.')


# ==============================================================
# 4. CARGAR PADDLEOCR
# ==============================================================

print('\nCargando PaddleOCR...')

ocr = PaddleOCR(
    lang='en',
    device='cpu',
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

print('PaddleOCR cargado correctamente.')


# ==============================================================
# 5. CARGAR SMOLVLM
# ==============================================================

print('\nCargando SmolVLM...')

MODELO_VLM = 'HuggingFaceTB/SmolVLM-Instruct'

processor_vlm = AutoProcessor.from_pretrained(
    MODELO_VLM
)

if DEVICE == 'cuda':

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

else:
    dtype = torch.float32


model_vlm = AutoModelForImageTextToText.from_pretrained(
    MODELO_VLM,
    torch_dtype=dtype
).to(DEVICE)

model_vlm.eval()

print('SmolVLM cargado correctamente.')


# ==============================================================
# 6. FUNCIONES AUXILIARES
# ==============================================================

def limpiar_texto(texto):

    if texto is None:
        return ''

    texto = str(texto).upper()

    return re.sub(
        r'[^A-Z0-9]',
        '',
        texto
    )


def sincronizar_gpu():

    if DEVICE == 'cuda':
        torch.cuda.synchronize()


# ==============================================================
# 7. FUNCION DE RECONOCIMIENTO CON PADDLEOCR
# ==============================================================

def reconocer_paddle(imagen):

    sincronizar_gpu()

    inicio = time.perf_counter()

    # Ampliar recorte
    imagen_ampliada = cv2.resize(
        imagen,
        None,
        fx=ESCALA_OCR,
        fy=ESCALA_OCR,
        interpolation=cv2.INTER_CUBIC
    )

    # Mejorar contraste
    gris = cv2.cvtColor(
        imagen_ampliada,
        cv2.COLOR_BGR2GRAY
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    gris = clahe.apply(gris)

    imagen_procesada = cv2.cvtColor(
        gris,
        cv2.COLOR_GRAY2BGR
    )

    # Reconocimiento
    resultados = ocr.predict(
        input=imagen_procesada
    )

    textos = []
    confianzas = []

    if resultados:

        for resultado in resultados:

            textos_detectados = resultado.get(
                'rec_texts', []
            )

            scores = resultado.get(
                'rec_scores', []
            )

            for texto, score in zip(
                textos_detectados,
                scores
            ):

                score = float(score)

                if score >= UMBRAL_OCR:

                    textos.append(str(texto))
                    confianzas.append(score)

    texto_final = limpiar_texto(
        ''.join(textos)
    )

    confianza = (
        float(np.mean(confianzas))
        if confianzas else 0.0
    )

    sincronizar_gpu()

    tiempo = time.perf_counter() - inicio

    return texto_final, confianza, tiempo


# ==============================================================
# 8. FUNCION DE RECONOCIMIENTO CON SMOLVLM
# ==============================================================

def reconocer_smolvlm(imagen):

    sincronizar_gpu()

    inicio = time.perf_counter()

    # Convertir recorte a RGB
    imagen_rgb = cv2.cvtColor(
        imagen,
        cv2.COLOR_BGR2RGB
    )

    imagen_pil = Image.fromarray(imagen_rgb)

    mensajes = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {
                    'type': 'text',
                    'text': (
                        'Read the vehicle license plate in this image. '
                        'Return only the complete alphanumeric plate. '
                        'Do not explain anything. '
                        'If the plate cannot be read, return UNKNOWN.'
                    )
                }
            ]
        }
    ]

    prompt = processor_vlm.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor_vlm(
        text=prompt,
        images=[imagen_pil],
        return_tensors='pt'
    ).to(DEVICE)

    with torch.inference_mode():

        salida = model_vlm.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    # Decodificar unicamente los tokens generados
    nuevos_tokens = salida[
        :,
        inputs['input_ids'].shape[1]:
    ]

    respuesta = processor_vlm.batch_decode(
        nuevos_tokens,
        skip_special_tokens=True
    )[0]

    respuesta = respuesta.strip()

    if 'UNKNOWN' in respuesta.upper():

        texto_final = ''

    else:

        texto_final = limpiar_texto(respuesta)

    sincronizar_gpu()

    tiempo = time.perf_counter() - inicio

    return texto_final, tiempo


# ==============================================================
# 9. BUSCAR IMAGENES DEL DATASET
# ==============================================================

print('\nBuscando imagenes...')

extensiones = [
    'jpg',
    'jpeg',
    'png',
    'bmp'
]

rutas_imagenes = []

for extension in extensiones:

    rutas_imagenes.extend(
        glob.glob(
            os.path.join(
                RUTA_DATASET,
                '**',
                'images',
                '*.' + extension
            ),
            recursive=True
        )
    )

rutas_imagenes = sorted(
    set(rutas_imagenes)
)

print('Imagenes encontradas:', len(rutas_imagenes))

if len(rutas_imagenes) == 0:

    raise Exception(
        'No se encontraron imagenes. Comprueba la estructura del dataset.'
    )


# ==============================================================
# 10. PREPARAR CSV DE RESULTADOS
# ==============================================================

registros = []

errores_ocr = 0

# Calentamiento de modelos.
# No incluimos estos tiempos en la comparativa.

print('\nCalentando modelos...')

imagen_prueba = np.ones(
    (100, 300, 3),
    dtype=np.uint8
) * 255

try:
    reconocer_paddle(imagen_prueba)
except Exception as e:
    print('Aviso calentamiento PaddleOCR:', e)

try:
    reconocer_smolvlm(imagen_prueba)
except Exception as e:
    print('Aviso calentamiento SmolVLM:', e)

print('Comenzando experimento...')


# ==============================================================
# 11. PROCESAR TODAS LAS IMAGENES
# ==============================================================

for idx, ruta_imagen in enumerate(
    tqdm(
        rutas_imagenes,
        desc='Comparando OCR y VLM'
    )
):

    imagen = cv2.imread(ruta_imagen)

    if imagen is None:
        continue

    alto, ancho = imagen.shape[:2]

    # ----------------------------------------------------------
    # A. DETECCION YOLO
    # ----------------------------------------------------------

    resultados = model_yolo(
        imagen,
        conf=UMBRAL_YOLO,
        verbose=False
    )

    detecciones = []

    for resultado in resultados:

        boxes = resultado.boxes.xyxy.cpu().numpy()

        scores = resultado.boxes.conf.cpu().numpy()

        for box, score in zip(boxes, scores):

            x1, y1, x2, y2 = map(int, box)

            x1 = max(0, x1)
            y1 = max(0, y1)

            x2 = min(ancho, x2)
            y2 = min(alto, y2)

            if x2 <= x1 or y2 <= y1:
                continue

            detecciones.append(
                (x1, y1, x2, y2, float(score))
            )

    # No usar la imagen completa como sustituto
    # cuando YOLO no detecte ninguna matricula.

    if not detecciones:
        continue

    # ----------------------------------------------------------
    # B. PROCESAR CADA MATRICULA DETECTADA
    # ----------------------------------------------------------

    for numero_det, deteccion in enumerate(detecciones):

        x1, y1, x2, y2, confianza_yolo = deteccion

        recorte = imagen[
            y1:y2,
            x1:x2
        ].copy()

        if recorte.size == 0:
            continue

        # --------------------------------------
        # PADDLEOCR
        # --------------------------------------

        try:

            texto_paddle, confianza_paddle, tiempo_paddle = (
                reconocer_paddle(recorte)
            )

        except Exception as e:

            print(
                '\nError PaddleOCR:',
                os.path.basename(ruta_imagen),
                e
            )

            texto_paddle = ''
            confianza_paddle = 0.0
            tiempo_paddle = np.nan

            errores_ocr += 1

        # --------------------------------------
        # SMOLVLM
        # --------------------------------------

        try:

            texto_smol, tiempo_smol = reconocer_smolvlm(
                recorte
            )

        except Exception as e:

            print(
                '\nError SmolVLM:',
                os.path.basename(ruta_imagen),
                e
            )

            texto_smol = ''
            tiempo_smol = np.nan

            errores_ocr += 1

            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

        # --------------------------------------
        # GUARDAR RESULTADOS
        # --------------------------------------

        registros.append({

            'imagen': os.path.relpath(
                ruta_imagen,
                RUTA_DATASET
            ),

            'deteccion': numero_det,

            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,

            'confianza_yolo': round(
                confianza_yolo, 4
            ),

            # Se rellenara con la matricula verdadera
            'matricula_real': '',

            'paddleocr_texto': texto_paddle,

            'paddleocr_confianza': round(
                confianza_paddle, 4
            ),

            'paddleocr_tiempo_s': (
                round(tiempo_paddle, 4)
                if np.isfinite(tiempo_paddle)
                else np.nan
            ),

            'smolvlm_texto': texto_smol,

            'smolvlm_tiempo_s': (
                round(tiempo_smol, 4)
                if np.isfinite(tiempo_smol)
                else np.nan
            )

        })

    # ----------------------------------------------------------
    # C. GUARDADO PERIODICO
    # ----------------------------------------------------------

    if (idx + 1) % 10 == 0 and registros:

        pd.DataFrame(registros).to_csv(
            RUTA_CSV,
            index=False,
            encoding='utf-8-sig'
        )

    gc.collect()


# ==============================================================
# 12. GUARDAR CSV FINAL
# ==============================================================

if not registros:

    raise Exception(
        'No se han detectado matriculas en las imagenes.'
    )

df = pd.DataFrame(registros)

df.to_csv(
    RUTA_CSV,
    index=False,
    encoding='utf-8-sig'
)

print('\n' + '=' * 60)
print('EXPERIMENTO FINALIZADO')
print('=' * 60)

print('Imagenes encontradas:', len(rutas_imagenes))
print('Matriculas detectadas:', len(df))
print('Errores de inferencia:', errores_ocr)

print(
    'Tiempo medio PaddleOCR:',
    round(df['paddleocr_tiempo_s'].mean(), 4),
    'segundos'
)

print(
    'Tiempo medio SmolVLM:',
    round(df['smolvlm_tiempo_s'].mean(), 4),
    'segundos'
)

print('\nCSV guardado:', RUTA_CSV)


# ==============================================================
# 13. CARGAR MATRICULAS REALES SI EXISTE ANOTACION
# ==============================================================

if RUTA_GROUND_TRUTH is not None:

    if os.path.exists(RUTA_GROUND_TRUTH):

        df_real = pd.read_csv(
            RUTA_GROUND_TRUTH,
            dtype=str
        )

        columnas_necesarias = [
            'imagen',
            'matricula_real'
        ]

        if not all(
            columna in df_real.columns
            for columna in columnas_necesarias
        ):

            raise ValueError(
                'El CSV de referencia necesita las columnas '
                'imagen y matricula_real.'
            )

        df = df.drop(
            columns=['matricula_real']
        ).merge(
            df_real[columnas_necesarias],
            on='imagen',
            how='left'
        )

        df['matricula_real'] = df[
            'matricula_real'
        ].fillna('')

        df.to_csv(
            RUTA_CSV,
            index=False,
            encoding='utf-8-sig'
        )

        print('Matriculas reales incorporadas.')

    else:

        print('No se encuentra el CSV de referencia.')


# ==============================================================
# 14. CALCULAR EXACTITUD Y CER
# ==============================================================

def distancia_levenshtein(a, b):

    fila = list(range(len(b) + 1))

    for i, ca in enumerate(a, 1):

        nueva = [i]

        for j, cb in enumerate(b, 1):

            nueva.append(
                min(
                    nueva[j - 1] + 1,
                    fila[j] + 1,
                    fila[j - 1] + (ca != cb)
                )
            )

        fila = nueva

    return fila[-1]


def evaluar_modelo(datos, columna_texto, columna_tiempo):

    aciertos = 0
    errores = 0
    total_caracteres = 0

    for _, fila in datos.iterrows():

        real = limpiar_texto(
            fila['matricula_real']
        )

        predicho = limpiar_texto(
            fila[columna_texto]
        )

        if real == predicho:
            aciertos += 1

        errores += distancia_levenshtein(
            real,
            predicho
        )

        total_caracteres += len(real)

    exactitud = (
        aciertos / len(datos) * 100
    )

    cer = (
        errores / total_caracteres * 100
        if total_caracteres > 0
        else np.nan
    )

    tiempo = datos[columna_tiempo].mean()

    return {
        'Exactitud (%)': exactitud,
        'CER (%)': cer,
        'Tiempo medio (s)': tiempo
    }


# ==============================================================
# 15. GENERAR METRICAS Y GRAFICAS
# ==============================================================

os.makedirs(
    RUTA_GRAFICAS,
    exist_ok=True
)

# Solo evaluar si disponemos de referencias reales.

evaluacion = df[
    df['matricula_real'].fillna('').astype(str).str.strip() != ''
].copy()

if len(evaluacion) > 0:

    resultados_metricas = {

        'PaddleOCR': evaluar_modelo(
            evaluacion,
            'paddleocr_texto',
            'paddleocr_tiempo_s'
        ),

        'SmolVLM': evaluar_modelo(
            evaluacion,
            'smolvlm_texto',
            'smolvlm_tiempo_s'
        )

    }

    tabla = pd.DataFrame(
        resultados_metricas
    ).T

    tabla.to_csv(
        RUTA_METRICAS,
        encoding='utf-8-sig'
    )

    print('\nRESULTADOS DE LA COMPARATIVA')
    print(tabla.round(4))

    # --------------------------------------
    # GRAFICA EXACTITUD
    # --------------------------------------

    ax = tabla['Exactitud (%)'].plot(
        kind='bar',
        title='Exactitud de reconocimiento'
    )

    ax.set_ylabel('Exactitud (%)')
    ax.set_xlabel('Modelo')
    plt.xticks(rotation=0)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            RUTA_GRAFICAS,
            'exactitud.png'
        )
    )

    plt.show()

    # --------------------------------------
    # GRAFICA CER
    # --------------------------------------

    ax = tabla['CER (%)'].plot(
        kind='bar',
        title='Character Error Rate'
    )

    ax.set_ylabel('CER (%)')
    ax.set_xlabel('Modelo')
    plt.xticks(rotation=0)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            RUTA_GRAFICAS,
            'cer.png'
        )
    )

    plt.show()

    # --------------------------------------
    # GRAFICA TIEMPO
    # --------------------------------------

    ax = tabla['Tiempo medio (s)'].plot(
        kind='bar',
        title='Tiempo medio de inferencia'
    )

    ax.set_ylabel('Segundos')
    ax.set_xlabel('Modelo')
    plt.xticks(rotation=0)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            RUTA_GRAFICAS,
            'tiempos.png'
        )
    )

    plt.show()

else:

    print('\nNo hay matriculas reales disponibles.')
    print('Se han guardado los tiempos y predicciones,')
    print('pero todavia no se puede calcular la exactitud ni el CER.')


# ==============================================================
# 16. DESCARGAR RESULTADOS
# ==============================================================

print('\nDescargando CSV...')

files.download(RUTA_CSV)

if len(evaluacion) > 0:

    files.download(RUTA_METRICAS)

    import shutil

    shutil.make_archive(
        '/content/graficas_comparativa',
        'zip',
        RUTA_GRAFICAS
    )

    files.download(
        '/content/graficas_comparativa.zip'
    )

print('\nPROCESO COMPLETADO')


VIDEO:

In [ ]:
!pip install -q paddlepaddle paddleocr ultralytics

# ============================================================
# PRACTICA 4 - DETECCION Y RECONOCIMIENTO DE MATRICULAS
# YOLO (best.pt) + PaddleOCR + OpenCV + CSV
# ============================================================

from google.colab import drive, files
drive.mount('/content/drive')

import cv2
import csv
import re
import time
import numpy as np
import os

from tqdm import tqdm
from ultralytics import YOLO
from paddleocr import PaddleOCR

# ============================================================
# 1. CONFIGURACION
# ============================================================

RUTA_VIDEO_ENTRADA = '/content/drive/MyDrive/C0142.MP4'
RUTA_MODELO_YOLO = '/content/drive/MyDrive/best.pt'

RUTA_VIDEO_SALIDA = '/content/video_resultado_ocr.mp4'
RUTA_CSV = '/content/matriculas_detectadas.csv'

UMBRAL_YOLO = 0.50
UMBRAL_OCR = 0.30

# Escala de ampliacion de las matriculas
ESCALA_OCR = 3

# ============================================================
# 2. COMPROBAR ARCHIVOS
# ============================================================

if not os.path.exists(RUTA_MODELO_YOLO):
    raise FileNotFoundError("No se encuentra best.pt en Google Drive.")

if not os.path.exists(RUTA_VIDEO_ENTRADA):
    raise FileNotFoundError("No se encuentra el video de entrada.")

# ============================================================
# 3. CARGAR MODELO YOLO
# ============================================================

print("Cargando modelo YOLO...")

# Este es TU modelo entrenado.
# Se utiliza exclusivamente para detectar matriculas.

model_yolo = YOLO(RUTA_MODELO_YOLO)

print("YOLO cargado correctamente.")

# ============================================================
# 4. CARGAR MODELO PADDLEOCR
# ============================================================

print("Cargando PaddleOCR...")


ocr = PaddleOCR(
    lang='en',
    device='cpu',
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)


print("PaddleOCR cargado correctamente.")

# ============================================================
# 5. FUNCION DE RECONOCIMIENTO DE MATRICULAS
# ============================================================

def reconocer_matricula(imagen):

    if imagen is None or imagen.size == 0:
        return "", 0.0

    # Ampliar el recorte para facilitar la lectura
    imagen = cv2.resize(
        imagen,
        None,
        fx=ESCALA_OCR,
        fy=ESCALA_OCR,
        interpolation=cv2.INTER_CUBIC
    )

    # Convertir a escala de grises
    gris = cv2.cvtColor(
        imagen,
        cv2.COLOR_BGR2GRAY
    )

    # Mejorar el contraste
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    gris = clahe.apply(gris)

    # PaddleOCR recibe la imagen en formato BGR
    imagen_procesada = cv2.cvtColor(
        gris,
        cv2.COLOR_GRAY2BGR
    )

    # Ejecutar reconocimiento OCR
    resultados = ocr.predict(input=imagen_procesada)

    textos = []
    confianzas = []

    if resultados:

        for resultado in resultados:

            # Obtener los textos y sus confianzas
            try:
                textos_detectados = resultado["rec_texts"]
                scores = resultado["rec_scores"]

            except (KeyError, TypeError):

                continue

            for texto, confianza in zip(
                textos_detectados,
                scores
            ):

                confianza = float(confianza)

                if confianza >= UMBRAL_OCR:

                    textos.append(str(texto))
                    confianzas.append(confianza)

    # Si no se reconoce ningun texto
    if not textos:
        return "", 0.0

    # Unir posibles fragmentos
    texto_final = "".join(textos)

    # Convertir a mayusculas y eliminar caracteres especiales
    texto_final = re.sub(
        r'[^A-Z0-9]',
        '',
        texto_final.upper()
    )

    if not texto_final:
        return "", 0.0

    confianza_final = float(np.mean(confianzas))

    return texto_final, confianza_final


# ============================================================
# 6. ABRIR VIDEO
# ============================================================

cap = cv2.VideoCapture(RUTA_VIDEO_ENTRADA)

if not cap.isOpened():
    raise Exception("No se ha podido abrir el video.")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30.0

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("\nInformacion del video:")
print(f"Resolucion: {width} x {height}")
print(f"FPS: {fps}")
print(f"Fotogramas: {total_frames}")

# ============================================================
# 7. CREAR VIDEO DE SALIDA
# ============================================================

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    RUTA_VIDEO_SALIDA,
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():
    raise Exception("No se ha podido crear el video de salida.")

# ============================================================
# 8. CREAR CSV
# ============================================================

archivo_csv = open(
    RUTA_CSV,
    'w',
    newline='',
    encoding='utf-8-sig'
)

writer = csv.writer(archivo_csv)

writer.writerow([
    'fotograma',
    'confianza_yolo',
    'matricula',
    'confianza_ocr'
])

# ============================================================
# 9. PROCESAMIENTO DEL VIDEO
# ============================================================

print("\nComenzando deteccion y reconocimiento...")

tiempo_inicio = time.time()

tiempo_total_yolo = 0.0
tiempo_total_ocr = 0.0

numero_detecciones = 0
numero_lecturas = 0
numero_frame = 0

with tqdm(
    total=total_frames if total_frames > 0 else None,
    desc="Procesando video"
) as barra:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # ----------------------------------------------------
        # A. DETECCION CON YOLO
        # ----------------------------------------------------

        inicio_yolo = time.time()

        resultados = model_yolo(
            frame,
            conf=UMBRAL_YOLO,
            verbose=False
        )

        tiempo_total_yolo += time.time() - inicio_yolo

        # ----------------------------------------------------
        # B. RECORRER LAS MATRICULAS DETECTADAS
        # ----------------------------------------------------

        for r in resultados:

            boxes = r.boxes.xyxy.cpu().numpy()
            scores = r.boxes.conf.cpu().numpy()

            for box, score in zip(boxes, scores):

                x1, y1, x2, y2 = map(int, box)

                # Limitar coordenadas a los bordes del video
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(width, x2)
                y2 = min(height, y2)

                if x2 <= x1 or y2 <= y1:
                    continue

                numero_detecciones += 1

                # ------------------------------------------------
                # C. RECORTAR LA MATRICULA
                # ------------------------------------------------

                recorte = frame[y1:y2, x1:x2]

                # ------------------------------------------------
                # D. RECONOCER TEXTO CON PADDLEOCR
                # ------------------------------------------------

                inicio_ocr = time.time()

                texto, confianza_ocr = reconocer_matricula(
                    recorte
                )

                tiempo_total_ocr += time.time() - inicio_ocr

                if texto != "":
                    numero_lecturas += 1
                else:
                    texto = "No leida"

                # ------------------------------------------------
                # E. DIBUJAR RECTANGULO
                # ------------------------------------------------

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )

                # ------------------------------------------------
                # F. DIBUJAR TEXTO RECONOCIDO
                # ------------------------------------------------

                etiqueta = texto

                tam_texto = cv2.getTextSize(
                    etiqueta,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    2
                )[0]

                # Evitar que el fondo salga fuera del fotograma
                alto_etiqueta = 35
                y_superior = max(0, y1 - alto_etiqueta)

                cv2.rectangle(
                    frame,
                    (x1, y_superior),
                    (min(width, x1 + tam_texto[0] + 10), y1),
                    (0, 255, 0),
                    -1
                )

                cv2.putText(
                    frame,
                    etiqueta,
                    (x1 + 5, max(20, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 0),
                    2,
                    cv2.LINE_AA
                )

                # ------------------------------------------------
                # G. GUARDAR DATOS EN CSV
                # ------------------------------------------------

                writer.writerow([
                    numero_frame,
                    round(float(score), 4),
                    texto,
                    round(float(confianza_ocr), 4)
                ])

        # ----------------------------------------------------
        # H. GUARDAR FOTOGRAMA EN EL VIDEO
        # ----------------------------------------------------

        out.write(frame)

        numero_frame += 1
        barra.update(1)

# ============================================================
# 10. FINALIZAR Y CERRAR ARCHIVOS
# ============================================================

cap.release()
out.release()

archivo_csv.close()

tiempo_final = time.time() - tiempo_inicio

# ============================================================
# 11. MOSTRAR ESTADISTICAS
# ============================================================

print("\n" + "=" * 50)
print("PROCESAMIENTO FINALIZADO")
print("=" * 50)

print(f"Fotogramas procesados: {numero_frame}")
print(f"Matriculas detectadas: {numero_detecciones}")
print(f"Matriculas leidas: {numero_lecturas}")

if numero_detecciones > 0:

    porcentaje_lectura = (
        numero_lecturas / numero_detecciones
    ) * 100

    print(f"Porcentaje de lectura: {porcentaje_lectura:.2f}%")

print(f"\nTiempo total: {tiempo_final:.2f} segundos")

if numero_frame > 0:
    print(f"FPS de procesamiento: {numero_frame / tiempo_final:.2f}")

if numero_detecciones > 0:

    print(
        f"Tiempo medio OCR por deteccion: "
        f"{tiempo_total_ocr / numero_detecciones:.4f} segundos"
    )

print(f"\nVideo: {RUTA_VIDEO_SALIDA}")
print(f"CSV: {RUTA_CSV}")

print("=" * 50)

# ============================================================
# 12. DESCARGAR RESULTADOS
# ============================================================

print("\nDescargando video...")
files.download(RUTA_VIDEO_SALIDA)

print("Descargando CSV...")
files.download(RUTA_CSV)

print("\nTodo terminado correctamente.")

